# Voice server for the VoxCPM Studio Android app

Runtime → Change runtime type → **T4 GPU** (or better). Pick the **MODEL** in the first cell, then run every cell.
The last cell prints a public `https://….trycloudflare.com` URL — paste it into the app's **Settings → Server URL** and tap **Test connection** (the app adapts its controls to the model).

**GPU limit reached?** Choose **CPU** as the runtime type instead and run the same cells. It works, but slowly (VoxCPM2: roughly 1–3 minutes per sentence; set **Advanced → Inference steps** to **4** in the app).

To switch models: **Runtime → Disconnect and delete runtime**, change MODEL, run all again.
When you're done, disconnect the same way so you don't use up your free GPU time while idle.

In [ ]:
# Which voice model should this server run?
MODEL = "VoxCPM2"  #@param ["VoxCPM2", "IndexTTS-2.5", "OmniVoice"]
# VoxCPM2:      default voice, voice design and cloning, 30 languages (~8 GB GPU memory).
# IndexTTS-2.5: voice cloning with emotions, English/Chinese/Japanese/Spanish/Arabic (~6 GB).
#               Needs a voice to clone (pick a file, record, or use My voices in the app).
# OmniVoice:    default voice, voice design (attributes) and cloning, 600+ languages (3.3 GB download).
# Optional: protect the server. Put the same key in the app's Settings.
API_KEY = ""  #@param {type:"string"}
assert MODEL in ("VoxCPM2", "IndexTTS-2.5", "OmniVoice"), MODEL

In [ ]:
def sh(cmd):
    """Runs a shell command and stops the notebook if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'Install step failed (exit {code}): {cmd}\nSee the messages above.')

get_ipython().system('nvidia-smi -L')
sh('wget -q -O server.py https://raw.githubusercontent.com/yassineallach/admin/claude/android-voxcpm-voice-app-8pkl1u/voxcpm-app/server/server.py')
sh('wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared')
if MODEL == "VoxCPM2":
    sh('pip -q install voxcpm fastapi "uvicorn[standard]" python-multipart soundfile librosa')
elif MODEL == "OmniVoice":
    sh('pip -q install omnivoice fastapi "uvicorn[standard]" python-multipart soundfile')
else:
    # IndexTTS-2.5 needs its own Python 3.11 environment; uv builds it inside index-tts/.venv (a few minutes).
    sh('pip -q install -U uv')
    sh('[ -d index-tts ] || git clone -q --depth 1 https://github.com/index-tts/index-tts.git')
    sh('cd index-tts && uv sync')
    # --python: install into IndexTTS's .venv, not the notebook's own (Kaggle's conda) environment.
    sh('cd index-tts && uv pip install --python .venv/bin/python fastapi "uvicorn[standard]" python-multipart soundfile')
    # ~5.5 GB of weights; helper models (~3 GB) download on the first start.
    sh('cd index-tts && .venv/bin/python -c "from huggingface_hub import snapshot_download; '
       'snapshot_download(\'IndexTeam/IndexTTS-2.5\', local_dir=\'checkpoints\')"')
if MODEL == "IndexTTS-2.5":
    sh('cd index-tts && .venv/bin/python -c "import fastapi, uvicorn, multipart, soundfile, indextts.infer_v2_5"')
print('Install done.')

In [ ]:
import json, os, subprocess, time, urllib.request, urllib.error
env = dict(os.environ, VOXCPM_API_KEY=API_KEY)
if MODEL == "VoxCPM2":
    cmd, cwd = ['python', os.path.abspath('server.py'), '--port', '8000'], None
elif MODEL == "OmniVoice":
    cmd, cwd = ['python', os.path.abspath('server.py'), '--engine', 'omnivoice', '--port', '8000'], None
else:
    # Run from the index-tts folder: IndexTTS reads some files from relative paths.
    cmd = ['.venv/bin/python', os.path.abspath('server.py'), '--engine', 'indextts', '--model-dir', 'checkpoints', '--port', '8000']
    cwd = 'index-tts'
server = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

def log_tail(n=40):
    return '\n'.join(open('server.log', errors='replace').read().splitlines()[-n:])

# Wait until the model has loaded (the first start also downloads it), up to 40 minutes.
headers = {'Authorization': f'Bearer {API_KEY}'} if API_KEY else {}
started, ready, last_note = time.time(), False, 0
while time.time() - started < 2400:
    if server.poll() is not None:
        raise RuntimeError('The server stopped while starting. Last lines of server.log:\n' + log_tail())
    try:
        urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/health', headers=headers), timeout=5)
        ready = True; break
    except urllib.error.HTTPError as e:
        if e.code == 500:
            raise RuntimeError(json.load(e).get('detail', 'Model failed to load') + '\n\nLast lines of server.log:\n' + log_tail())
        # 503: still loading
    except Exception:
        pass  # not listening yet
    if time.time() - last_note > 60:
        last_note = time.time()
        print(f'Loading {MODEL}… {int(time.time() - started) // 60} min')
    time.sleep(3)
if not ready:
    raise RuntimeError('The model is taking more than 40 minutes to load. Last lines of server.log:\n' + log_tail())
print(log_tail(15))
print(f'\n{MODEL} is ready.')

In [ ]:
import re, subprocess, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.DEVNULL, stderr=open('tunnel.log', 'w'))
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        print('Server URL for the app:', m.group(0)); break
    time.sleep(1)
else:
    print(open('tunnel.log').read())

### Troubleshooting
If the app shows an error, run the cell below and send its output.

In [ ]:
print('Server running:', server.poll() is None)
print(open('server.log', errors='replace').read()[-4000:])